# Q1: DCGAN vs WGAN-GP — Tackling Mode Collapse
**Course:** Generative AI (AI4009) | **Assignment:** 03 | **Platform:** Kaggle T4 x2

## Cell 1 — Imports

In [ ]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid, save_image
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

## Cell 2 — Hyperparameters

In [ ]:
IMG_SIZE      = 64
Z_DIM         = 100
CHANNELS      = 3
FEATURES_G    = 64
FEATURES_D    = 64
BATCH_SIZE    = 64
LR            = 0.0002
BETAS         = (0.5, 0.999)
EPOCHS_DCGAN  = 30
EPOCHS_WGAN   = 30
CRITIC_ITERS  = 5
LAMBDA_GP     = 10
SAVE_EVERY    = 5

os.makedirs('outputs/dcgan', exist_ok=True)
os.makedirs('outputs/wgan',  exist_ok=True)
os.makedirs('checkpoints',   exist_ok=True)
print('Config ready.')

## Cell 3 — Find Dataset Path

In [ ]:
def find_image_root():
    """
    Walks /kaggle/input and returns the first folder
    that contains image files — works for flat or nested datasets.
    """
    preferred = [
        '/kaggle/input/anime-faces',
        '/kaggle/input/pokemon-sprites',
    ]
    VALID = {'.png', '.jpg', '.jpeg', '.webp'}
    for base in preferred:
        if not os.path.exists(base):
            continue
        for root, dirs, files in os.walk(base):
            imgs = [f for f in files if os.path.splitext(f)[1].lower() in VALID]
            if imgs:
                print(f'Found {len(imgs)} images in: {root}')
                return root
    # fallback: scan all /kaggle/input
    for root, dirs, files in os.walk('/kaggle/input'):
        imgs = [f for f in files if os.path.splitext(f)[1].lower() in VALID]
        if len(imgs) > 100:
            print(f'Found {len(imgs)} images in: {root}')
            return root
    raise FileNotFoundError('No images found. Add a dataset to this notebook.')

DATA_PATH = find_image_root()
print(f'DATA_PATH = {DATA_PATH}')

## Cell 4 — Dataset & DataLoader

In [ ]:
class FlatImageDataset(Dataset):
    """Loads images from a flat folder — no subfolders required."""
    VALID_EXTS = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}

    def __init__(self, folder, transform=None):
        self.transform = transform
        self.paths = []
        for f in sorted(os.listdir(folder)):
            if os.path.splitext(f)[1].lower() in self.VALID_EXTS:
                self.paths.append(os.path.join(folder, f))
        if not self.paths:          # nested fallback
            for root, _, files in os.walk(folder):
                for f in files:
                    if os.path.splitext(f)[1].lower() in self.VALID_EXTS:
                        self.paths.append(os.path.join(root, f))
        assert self.paths, f'No images found in {folder}'
        print(f'Dataset ready — {len(self.paths)} images')

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), 0)
        if self.transform:
            img = self.transform(img)
        return img, 0   # dummy label


transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)   # [-1, 1]
])

full_dataset   = FlatImageDataset(DATA_PATH, transform=transform)
SUBSET_SIZE    = min(len(full_dataset), 20000)
subset_indices = torch.randperm(len(full_dataset))[:SUBSET_SIZE]
dataset        = torch.utils.data.Subset(full_dataset, subset_indices)

dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                        shuffle=True, num_workers=2,
                        pin_memory=True, drop_last=True)

print(f'Subset: {SUBSET_SIZE} | Batches/epoch: {len(dataloader)}')

## Cell 5 — Visualise Training Samples

In [ ]:
real_batch = next(iter(dataloader))
grid = make_grid(real_batch[0][:16], nrow=4, normalize=True, value_range=(-1,1))
plt.figure(figsize=(8,8))
plt.title('Sample Training Images', fontsize=14)
plt.imshow(grid.permute(1,2,0).numpy())
plt.axis('off'); plt.tight_layout()
plt.savefig('outputs/training_samples.png', dpi=100)
plt.show()

## Cell 6 — DCGAN Architecture

In [ ]:
def weights_init(m):
    cls = m.__class__.__name__
    if 'Conv' in cls:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in cls:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


class DCGANGenerator(nn.Module):
    def __init__(self, z=Z_DIM, ch=CHANNELS, f=FEATURES_G):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z,   f*8, 4,1,0,bias=False), nn.BatchNorm2d(f*8), nn.ReLU(True),
            nn.ConvTranspose2d(f*8, f*4, 4,2,1,bias=False), nn.BatchNorm2d(f*4), nn.ReLU(True),
            nn.ConvTranspose2d(f*4, f*2, 4,2,1,bias=False), nn.BatchNorm2d(f*2), nn.ReLU(True),
            nn.ConvTranspose2d(f*2, f,   4,2,1,bias=False), nn.BatchNorm2d(f),   nn.ReLU(True),
            nn.ConvTranspose2d(f,   ch,  4,2,1,bias=False), nn.Tanh()
        )
    def forward(self, z): return self.net(z)


class DCGANDiscriminator(nn.Module):
    """NO Sigmoid — we use BCEWithLogitsLoss which is autocast-safe."""
    def __init__(self, ch=CHANNELS, f=FEATURES_D):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch,  f,   4,2,1,bias=False), nn.LeakyReLU(0.2,True),
            nn.Conv2d(f,   f*2, 4,2,1,bias=False), nn.BatchNorm2d(f*2), nn.LeakyReLU(0.2,True),
            nn.Conv2d(f*2, f*4, 4,2,1,bias=False), nn.BatchNorm2d(f*4), nn.LeakyReLU(0.2,True),
            nn.Conv2d(f*4, f*8, 4,2,1,bias=False), nn.BatchNorm2d(f*8), nn.LeakyReLU(0.2,True),
            nn.Conv2d(f*8, 1,   4,1,0,bias=False)
            # No Sigmoid — BCEWithLogitsLoss handles it
        )
    def forward(self, x): return self.net(x).view(-1)


dcgan_G = DCGANGenerator().to(device)
dcgan_D = DCGANDiscriminator().to(device)

if torch.cuda.device_count() > 1:
    dcgan_G = nn.DataParallel(dcgan_G)
    dcgan_D = nn.DataParallel(dcgan_D)
    print(f'DataParallel on {torch.cuda.device_count()} GPUs')

dcgan_G.apply(weights_init)
dcgan_D.apply(weights_init)
print('DCGAN models initialised.')

## Cell 7 — WGAN-GP Architecture

In [ ]:
class WGANGenerator(nn.Module):
    def __init__(self, z=Z_DIM, ch=CHANNELS, f=FEATURES_G):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z,   f*8, 4,1,0,bias=False), nn.BatchNorm2d(f*8), nn.ReLU(True),
            nn.ConvTranspose2d(f*8, f*4, 4,2,1,bias=False), nn.BatchNorm2d(f*4), nn.ReLU(True),
            nn.ConvTranspose2d(f*4, f*2, 4,2,1,bias=False), nn.BatchNorm2d(f*2), nn.ReLU(True),
            nn.ConvTranspose2d(f*2, f,   4,2,1,bias=False), nn.BatchNorm2d(f),   nn.ReLU(True),
            nn.ConvTranspose2d(f,   ch,  4,2,1,bias=False), nn.Tanh()
        )
    def forward(self, z): return self.net(z)


class WGANCritic(nn.Module):
    """No Sigmoid, no BatchNorm — uses InstanceNorm for WGAN-GP stability."""
    def __init__(self, ch=CHANNELS, f=FEATURES_D):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch,  f,   4,2,1,bias=False), nn.LeakyReLU(0.2,True),
            nn.Conv2d(f,   f*2, 4,2,1,bias=False), nn.InstanceNorm2d(f*2,affine=True), nn.LeakyReLU(0.2,True),
            nn.Conv2d(f*2, f*4, 4,2,1,bias=False), nn.InstanceNorm2d(f*4,affine=True), nn.LeakyReLU(0.2,True),
            nn.Conv2d(f*4, f*8, 4,2,1,bias=False), nn.InstanceNorm2d(f*8,affine=True), nn.LeakyReLU(0.2,True),
            nn.Conv2d(f*8, 1,   4,1,0,bias=False)
        )
    def forward(self, x): return self.net(x).view(-1)


def compute_gradient_penalty(critic, real, fake, device):
    B = real.size(0)
    alpha = torch.rand(B,1,1,1, device=device).expand_as(real)
    interp = (alpha * real + (1-alpha) * fake).requires_grad_(True)
    score  = critic(interp)
    grads  = autograd.grad(
        outputs=score, inputs=interp,
        grad_outputs=torch.ones_like(score),
        create_graph=True, retain_graph=True, only_inputs=True
    )[0]
    grads = grads.view(B, -1)
    return ((grads.norm(2, dim=1) - 1) ** 2).mean()


wgan_G = WGANGenerator().to(device)
wgan_C = WGANCritic().to(device)

if torch.cuda.device_count() > 1:
    wgan_G = nn.DataParallel(wgan_G)
    wgan_C = nn.DataParallel(wgan_C)

wgan_G.apply(weights_init)
wgan_C.apply(weights_init)
print('WGAN-GP models initialised.')

## Cell 8 — Train DCGAN

In [ ]:
opt_dcgan_G = optim.Adam(dcgan_G.parameters(), lr=LR, betas=BETAS)
opt_dcgan_D = optim.Adam(dcgan_D.parameters(), lr=LR, betas=BETAS)

# BCEWithLogitsLoss = Sigmoid + BCE — autocast safe, no Sigmoid needed in model
criterion = nn.BCEWithLogitsLoss()

fixed_noise = torch.randn(16, Z_DIM, 1, 1, device=device)
scaler      = torch.amp.GradScaler('cuda')

dcgan_G_losses, dcgan_D_losses = [], []

print('Starting DCGAN Training...')
print('=' * 60)

for epoch in range(1, EPOCHS_DCGAN + 1):
    ep_G, ep_D = 0.0, 0.0
    loop = tqdm(dataloader, desc=f'[DCGAN] Epoch {epoch}/{EPOCHS_DCGAN}', leave=False)

    for real_imgs, _ in loop:
        real_imgs = real_imgs.to(device)
        B = real_imgs.size(0)
        real_lbl = torch.ones(B,  device=device)
        fake_lbl = torch.zeros(B, device=device)

        # ── Discriminator ────────────────────────────────
        opt_dcgan_D.zero_grad()
        with torch.amp.autocast('cuda'):
            loss_D_real = criterion(dcgan_D(real_imgs), real_lbl)
            noise       = torch.randn(B, Z_DIM, 1, 1, device=device)
            fake_imgs   = dcgan_G(noise).detach()
            loss_D_fake = criterion(dcgan_D(fake_imgs), fake_lbl)
            loss_D      = loss_D_real + loss_D_fake
        scaler.scale(loss_D).backward()
        scaler.step(opt_dcgan_D)

        # ── Generator ────────────────────────────────────
        opt_dcgan_G.zero_grad()
        with torch.amp.autocast('cuda'):
            noise     = torch.randn(B, Z_DIM, 1, 1, device=device)
            fake_imgs = dcgan_G(noise)
            loss_G    = criterion(dcgan_D(fake_imgs), real_lbl)
        scaler.scale(loss_G).backward()
        scaler.step(opt_dcgan_G)
        scaler.update()

        ep_G += loss_G.item(); ep_D += loss_D.item()
        loop.set_postfix(G=f'{loss_G.item():.4f}', D=f'{loss_D.item():.4f}')

    avg_G = ep_G / len(dataloader)
    avg_D = ep_D / len(dataloader)
    dcgan_G_losses.append(avg_G)
    dcgan_D_losses.append(avg_D)
    print(f'[DCGAN] Epoch {epoch:03d}/{EPOCHS_DCGAN} | G: {avg_G:.4f} | D: {avg_D:.4f}')

    with torch.no_grad():
        fake = dcgan_G(fixed_noise).cpu()
    save_image(fake, f'outputs/dcgan/epoch_{epoch:03d}.png',
               normalize=True, value_range=(-1,1), nrow=4)

    if epoch % SAVE_EVERY == 0:
        torch.save({'epoch':epoch,'G':dcgan_G.state_dict(),'D':dcgan_D.state_dict(),
                    'G_loss':dcgan_G_losses,'D_loss':dcgan_D_losses},
                   f'checkpoints/dcgan_{epoch:03d}.pth')

print('DCGAN Training Complete!')

## Cell 9 — Train WGAN-GP

In [ ]:
opt_wgan_G = optim.Adam(wgan_G.parameters(), lr=LR, betas=BETAS)
opt_wgan_C = optim.Adam(wgan_C.parameters(), lr=LR, betas=BETAS)

fixed_noise_w = torch.randn(16, Z_DIM, 1, 1, device=device)

wgan_G_losses, wgan_C_losses, wgan_GP_losses = [], [], []

print('Starting WGAN-GP Training...')
print('=' * 60)

for epoch in range(1, EPOCHS_WGAN + 1):
    ep_G, ep_C, ep_GP, gen_steps = 0.0, 0.0, 0.0, 0
    loop = tqdm(dataloader, desc=f'[WGAN-GP] Epoch {epoch}/{EPOCHS_WGAN}', leave=False)

    for i, (real_imgs, _) in enumerate(loop):
        real_imgs = real_imgs.to(device)
        B = real_imgs.size(0)

        # ── Critic ───────────────────────────────────────
        opt_wgan_C.zero_grad()
        noise     = torch.randn(B, Z_DIM, 1, 1, device=device)
        fake_imgs = wgan_G(noise).detach()
        gp        = compute_gradient_penalty(wgan_C, real_imgs, fake_imgs, device)
        loss_C    = wgan_C(fake_imgs).mean() - wgan_C(real_imgs).mean() + LAMBDA_GP * gp
        loss_C.backward()
        opt_wgan_C.step()
        ep_C += loss_C.item(); ep_GP += gp.item()

        # ── Generator (every CRITIC_ITERS steps) ─────────
        loss_G = torch.tensor(0.0)
        if i % CRITIC_ITERS == 0:
            opt_wgan_G.zero_grad()
            noise     = torch.randn(B, Z_DIM, 1, 1, device=device)
            fake_imgs = wgan_G(noise)
            loss_G    = -wgan_C(fake_imgs).mean()
            loss_G.backward()
            opt_wgan_G.step()
            ep_G += loss_G.item(); gen_steps += 1

        loop.set_postfix(G=f'{loss_G.item():.4f}', C=f'{loss_C.item():.4f}', GP=f'{gp.item():.4f}')

    avg_G  = ep_G  / max(gen_steps, 1)
    avg_C  = ep_C  / len(dataloader)
    avg_GP = ep_GP / len(dataloader)
    wgan_G_losses.append(avg_G)
    wgan_C_losses.append(avg_C)
    wgan_GP_losses.append(avg_GP)
    print(f'[WGAN-GP] Epoch {epoch:03d}/{EPOCHS_WGAN} | G: {avg_G:.4f} | C: {avg_C:.4f} | GP: {avg_GP:.4f}')

    with torch.no_grad():
        fake = wgan_G(fixed_noise_w).cpu()
    save_image(fake, f'outputs/wgan/epoch_{epoch:03d}.png',
               normalize=True, value_range=(-1,1), nrow=4)

    if epoch % SAVE_EVERY == 0:
        torch.save({'epoch':epoch,'G':wgan_G.state_dict(),'C':wgan_C.state_dict(),
                    'G_loss':wgan_G_losses,'C_loss':wgan_C_losses},
                   f'checkpoints/wgan_{epoch:03d}.pth')

print('WGAN-GP Training Complete!')

## Cell 10 — Loss Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
ep_d = range(1, len(dcgan_G_losses)+1)
axes[0].plot(ep_d, dcgan_G_losses, label='Generator',     color='#e74c3c', lw=2)
axes[0].plot(ep_d, dcgan_D_losses, label='Discriminator', color='#3498db', lw=2)
axes[0].set_title('DCGAN Losses', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

ep_w = range(1, len(wgan_G_losses)+1)
axes[1].plot(ep_w, wgan_G_losses,  label='Generator',        color='#e74c3c', lw=2)
axes[1].plot(ep_w, wgan_C_losses,  label='Critic',           color='#2ecc71', lw=2)
axes[1].plot(ep_w, wgan_GP_losses, label='Gradient Penalty', color='#9b59b6', lw=2, ls='--')
axes[1].set_title('WGAN-GP Losses', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/training_losses.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 11 — Generated Samples (10 each)

In [ ]:
def show_samples(gen, title, n=10):
    gen.eval()
    with torch.no_grad():
        imgs = gen(torch.randn(n, Z_DIM, 1, 1, device=device)).cpu()
    grid = make_grid(imgs, nrow=5, normalize=True, value_range=(-1,1))
    plt.figure(figsize=(14,6))
    plt.title(title, fontsize=15, fontweight='bold')
    plt.imshow(grid.permute(1,2,0).numpy()); plt.axis('off')
    plt.tight_layout()
    fname = title.lower().replace(' ','_').replace('/','_').replace('—','') + '.png'
    plt.savefig(f'outputs/{fname}', dpi=150); plt.show()
    gen.train()

show_samples(dcgan_G, 'DCGAN — Generated Samples')
show_samples(wgan_G,  'WGAN-GP — Generated Samples')

## Cell 12 — Side-by-Side Comparison

In [ ]:
torch.manual_seed(99)
shared_noise = torch.randn(5, Z_DIM, 1, 1, device=device)
with torch.no_grad():
    d_out = dcgan_G(shared_noise).cpu()
    w_out = wgan_G(shared_noise).cpu()

fig, axes = plt.subplots(2, 5, figsize=(16,7))
fig.suptitle('DCGAN (top) vs WGAN-GP (bottom)', fontsize=14, fontweight='bold')
for col in range(5):
    for row, imgs in enumerate([d_out, w_out]):
        img = imgs[col].permute(1,2,0).numpy()
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        axes[row,col].imshow(img)
        axes[row,col].axis('off')
        axes[row,col].set_title(['DCGAN','WGAN-GP'][row]+f' #{col+1}', fontsize=9)
plt.tight_layout()
plt.savefig('outputs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 13 — Save Final Weights + Loss History

In [ ]:
torch.save(dcgan_G.state_dict(), 'outputs/dcgan_generator_final.pth')
torch.save(wgan_G.state_dict(),  'outputs/wgan_generator_final.pth')

with open('outputs/loss_history.pkl', 'wb') as f:
    pickle.dump({
        'dcgan_G': dcgan_G_losses, 'dcgan_D': dcgan_D_losses,
        'wgan_G' : wgan_G_losses,  'wgan_C' : wgan_C_losses,
        'wgan_GP': wgan_GP_losses
    }, f)

print('Saved:')
for f in sorted(os.listdir('outputs')): print(f'  {f}')